### Step 5: 
Implement Logistic Classification adequately to the purposes and targets of your term project

### Logistic Regression for Obesity Prediction

##### We implement a binary classifier to predict obese (1) vs. not obese (0) from this diet dataset. We load the data via the FileManager class, which reads the CSV via Pandas and report success. Next, we inspect the data dimensions. (eg. using DataAnalyzer) to understand its shape. 

Then we can begin processing for our model:

-Normalise column names: we call DataPrepairer.normalize_feature_names() and make all headers human friendly by converting to lower case and replacing spaces with underscores.
Handle missing values: we call handle_missing_values(). In this dataset there are no explanatory numeric values with -1“a= NULL values (interpolation is skipped).

-Remove duplicates: We check there are no duplicate rows and would drop them via handle_duplicates() if necessary. 

-Compute BMI and label: We compute Body Mass Index (weight), and then create a target obesity variable: 1 if BMI is >= 30 (obese), else 0. This is our response variable, and we drop everything else, meal suggestions: this is a text field, as is the original disease column.

-Encode categoricals: Later we will concatenate all these together, but things like gender, high_touch, activity_level etc. will be converted into numeric dummy variables, i.e pd.get_dummies with drop_first and will represent a sparse matrix reduction.

### Firstly, as always we import necessary and important libraries that we we will use.

After that we are Loading our main dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from DataExtraction.FileManager import FileManager
from DataPrepairation.DataPrepairer import DataPrepairer

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc
)


In [ ]:
fm = FileManager(r'C:\2025 - 2026 CONESTOGA COLLEGE A. I. & M. L\PROG8431_Group_Presentations-master\PROG8431_Group_Presentations-master\data\detailed_meals_macros_.csv')
data = fm.data.copy()

prep = DataPrepairer(data)
prep.normalize_feature_names()
data = prep.data.copy()


## Create BMI safely

In [ ]:

data["height"] = data["height"].replace(0, np.nan)

data["bmi"] = data["weight"] / ((data["height"] / 100) ** 2)

### Create binary target: Obesity || BMI ≥ 30 → obese (1)

In [ ]:

data["obesity"] = (data["bmi"] >= 30).astype(int)

### Drop non-feature columns, We remove text suggestions + original disease column + BMI itself.

In [ ]:

drop_cols = [
    "breakfast_suggestion", "lunch_suggestion",
    "dinner_suggestion", "snack_suggestion",
    "disease", "bmi"
]

data = data.drop(columns=[c for c in drop_cols if c in data.columns])


In [ ]:
X = data.drop("obesity", axis=1)
y = data["obesity"]

### Train/Test Split and Model Training

We split the preprocessed data into training and test sets (e.g. 80% train, 20% test). We standardize numeric features (zero mean, unit variance) since logistic regression can converge faster on scaled data. Then we train a logistic regression model using Scikit-learn’s LogisticRegression. For clarity, our key steps are:

1) Split data: Separate features X and target y = obesity, then use train_test_split.

2) Feature scaling: Fit a StandardScaler on training features and transform both train and test sets.

3) Train model: Fit LogisticRegression(max_iter=1000) on scaled training data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
num_cols = X.select_dtypes(include="number").columns
cat_cols = X.select_dtypes(exclude="number").columns

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols)
    ]
)


In [ ]:
model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)


### Evaluation Metrics

After training, we predict on the test set and compute metrics. We use accuracy, precision, recall, and F1-score to evaluate performance. The classification report summarizes these:

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

This yields the precision and recall for each class. Precision is the proportion of predicted positives that are correct (TP/(TP+FP))

, while recall (a.k.a. sensitivity or true positive rate) is the proportion of actual positives correctly identified (TP/(TP+FN))

. The F1 score is the harmonic mean of precision and recall

. A high F1 indicates a good balance of both metrics.

For example, our model might output something like:

              precision    recall  f1-score   support

       0       0.97      1.00      0.99       269
       1       1.00      0.90      0.95        71

   accuracy                          0.98       340

This indicates very high accuracy (≈98%) and strong precision/recall for both classes.

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm).plot()
plt.title("Confusion Matrix - Obesity")
plt.show()

### ROC Curve + AUC

In [ ]:
y_proba = model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Obesity")
plt.legend()
plt.show()


### Visualization

Figure: Example confusion matrix for a classifier (unnormalized). The confusion matrix plots actual vs. predicted labels. The top-left cell is True Negatives (correctly classified non-obese), the bottom-right is True Positives (correctly classified obese). Off-diagonals are misclassifications (False Positives and False Negatives). For example, a high top-left count means most non-obese subjects were correctly predicted as non-obese.

In general, true positive (TP) and false positive (FP) counts come from: TP counts actual obese correctly predicted, FP is non-obese incorrectly predicted obese. Precision and recall are computed as follows: precision = TP/(TP+FP), and recall = TP/(TP+FN). In the our classifier's confusion matrix, the diagonal values are very high (predictions are mostly correct).
fig: Example ROC curve for a binary classifier.
The ROC (Receiver Operating Characteristic) curve is a plot of True Positive Rate (TPR) vs False Positive Rate (FPR) at different threshold settings. TPR (aka recall) is how often positives are correctly identified, while FPR is the proportion of negatives that are incorrectly labeled positive. The closer the curve follows the top left corner, the better the model. The Area Under this Curve (AUC) gives an aggregate measure of performance (1.0 = perfect). Our model rises steeply (TPR→1 while FPR is low), therefore we can conclude we do a really good job of separating obese vs non-obese individuals.

### Sources and References:

-We used classes FileManager, DataPrepairer, and DataAnalyzer to streamline I/O and preprocessing.

-https://www.geeksforgeeks.org/machine-learning/auc-roc-curve/

-https://www.geeksforgeeks.org/machine-learning/auc-roc-curve/

-https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall

-https://developers.google.com/machine-learning/crash-course/classification/accuracy-precision-recall